In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. GLIDER FLIGHT PROFILES & LOGGER ENGINE
# ==========================================
FLIGHT_PROFILES = {
    "Hand Launch & Thermal Glide": {
        "sink_rate_mps": 0.65,
        "cruise_mps": 9.5,
        "thermal_boost": 3.5,
        "desc": "Smooth hand launch followed by a mild thermal lift pocket and steady glide descent."
    },
    "Steep Dive & Pull-Up": {
        "sink_rate_mps": 1.15,
        "cruise_mps": 12.5,
        "thermal_boost": 0.0,
        "desc": "Launch, controlled nose-down dive with higher speed, and a noticeable pull-up g-load peak."
    },
    "Gusty Ridge Glide": {
        "sink_rate_mps": 0.85,
        "cruise_mps": 10.5,
        "thermal_boost": 1.8,
        "desc": "Ridge-lift glide experiencing periodic wind gusts and dynamic acceleration fluctuations."
    },
}

def generate_glider_dataset(profile_name, fs_hz, duration_s, launch_alt_m, noise_level, seed=42):
    prof = FLIGHT_PROFILES[profile_name]
    n_samples = max(int(fs_hz * duration_s), 50)
    t = np.linspace(0.0, duration_s, n_samples)
    dt = t[1] - t[0]

    # Forward speed & horizontal distance
    v_fwd = prof["cruise_mps"] + 1.2 * np.sin(0.5 * np.pi * t / max(duration_s, 1.0))
    if profile_name == "Steep Dive & Pull-Up":
        dive_window = np.exp(-((t - 0.55 * duration_s) / (0.12 * duration_s))**2)
        v_fwd = v_fwd + 6.0 * dive_window
    x_dist = np.cumsum(v_fwd) * dt

    # Clean altitude profile [m]
    climb_phase = 2.5 * (1.0 - np.exp(-t / 1.5))
    sink_loss = prof["sink_rate_mps"] * t
    if profile_name == "Hand Launch & Thermal Glide":
        thermal = prof["thermal_boost"] * np.exp(-((t - 0.45 * duration_s) / (0.18 * duration_s))**2)
    elif profile_name == "Steep Dive & Pull-Up":
        dive_dip = -7.0 * np.exp(-((t - 0.52 * duration_s) / (0.10 * duration_s))**2)
        pull_lift = +2.5 * np.exp(-((t - 0.65 * duration_s) / (0.10 * duration_s))**2)
        thermal = dive_dip + pull_lift
    else:  # "Gusty Ridge Glide"
        thermal = prof["thermal_boost"] * np.sin(np.pi * t / max(duration_s, 1.0)) + 0.8 * np.sin(2.2 * t)

    alt_clean = np.maximum(launch_alt_m + climb_phase - sink_loss + thermal, 0.0)

    # Vertical velocity & vertical acceleration [g]
    vz = np.gradient(alt_clean, dt)
    az_mps2 = np.gradient(vz, dt)
    a_vert_g = 1.0 + (az_mps2 / 9.81)

    if profile_name == "Steep Dive & Pull-Up":
        a_vert_g += 1.45 * np.exp(-((t - 0.63 * duration_s) / (0.07 * duration_s))**2)
    elif profile_name == "Gusty Ridge Glide":
        a_vert_g += 0.35 * np.sin(3.5 * t) + 0.22 * np.cos(7.1 * t)

    # Sensor noise
    rng = np.random.default_rng(seed)
    alt_meas = np.maximum(alt_clean + rng.normal(0.0, 0.35 * noise_level, size=n_samples), 0.0)
    accel_g_meas = a_vert_g + rng.normal(0.0, 0.06 * noise_level, size=n_samples)

    df = pd.DataFrame({
        "time_s": np.round(t, 3),
        "distance_m": np.round(x_dist, 2),
        "altitude_m": np.round(alt_meas, 2),
        "accel_vert_g": np.round(accel_g_meas, 3),
    })

    return df, {
        "max_alt_m": float(np.max(alt_meas)),
        "final_alt_m": float(alt_meas[-1]),
        "total_dist_m": float(x_dist[-1]),
        "max_g": float(np.max(accel_g_meas)),
        "min_g": float(np.min(accel_g_meas)),
        "rms_g": float(np.sqrt(np.mean(accel_g_meas**2))),
        "samples": n_samples,
    }

# ==========================================
# 2. 2D FLIGHT PATH + SELECTED DATASET PLOT
# ==========================================
def draw_glider_rig_and_dataset(profile_name, dataset_mode, fs_hz, duration_s, launch_alt_m, noise_level, run_idx):
    df, stats = generate_glider_dataset(
        profile_name, fs_hz, duration_s, launch_alt_m, noise_level, seed=42 + run_idx
    )
    csv_filename = f"glider_flight_{profile_name.lower().replace(' ', '_').replace('&', 'and')}.csv"
    df.to_csv(csv_filename, index=False)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_path, ax_ts) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.05, 1.4]})

    # --- LEFT: 2D Glider Flight Profile (Distance vs. Altitude) ---
    ax_path.plot(df["distance_m"], df["altitude_m"], color="#2563eb", lw=2.3, label="Glider Trajectory")
    ax_path.fill_between(df["distance_m"], 0.0, df["altitude_m"], color="#93c5fd", alpha=0.25)
    ax_path.scatter([df["distance_m"].iloc[0]], [df["altitude_m"].iloc[0]], color="#16a34a", s=65, zorder=6, label="Launch")
    ax_path.scatter([df["distance_m"].iloc[-1]], [df["altitude_m"].iloc[-1]], color="#dc2626", s=65, zorder=6, label="End / Touchdown")

    ax_path.set_title(f"2D Flight Profile — {profile_name}", fontsize=11, fontweight="bold", pad=10)
    ax_path.set_xlabel("Downrange Distance [m]", fontsize=9.5)
    ax_path.set_ylabel("Altitude [m]", fontsize=9.5)
    ax_path.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Selected Flight Dataset Time Series ---
    if dataset_mode == "Altitude Dataset [m]":
        ax_ts.plot(df["time_s"], df["altitude_m"], color="#0284c7", lw=2.0, label="Logged Altitude [m]")
        pk_idx = int(np.argmax(df["altitude_m"]))
        ax_ts.scatter([df["time_s"].iloc[pk_idx]], [df["altitude_m"].iloc[pk_idx]], color="#dc2626", s=70, zorder=6)
        ax_ts.annotate(
            f"Peak Alt: {stats['max_alt_m']:.1f} m @ {df['time_s'].iloc[pk_idx]:.1f} s",
            xy=(df["time_s"].iloc[pk_idx], df["altitude_m"].iloc[pk_idx]),
            xytext=(10, -18), textcoords="offset points",
            fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#f0f9ff", edgecolor="#0284c7")
        )
        ax_ts.set_ylabel("Altitude [m]", fontsize=9.5)
        ax_ts.set_title(f"Flight Dataset: Altitude vs. Time ({fs_hz:.0f} Hz)", fontsize=11, fontweight="bold", pad=10)
    else:
        ax_ts.plot(df["time_s"], df["accel_vert_g"], color="#ea580c", lw=1.8, label="Vertical Acceleration [g]")
        ax_ts.axhline(1.0, color="#64748b", ls="--", lw=1.1, label="1.0 g Level")
        pk_idx = int(np.argmax(df["accel_vert_g"]))
        ax_ts.scatter([df["time_s"].iloc[pk_idx]], [df["accel_vert_g"].iloc[pk_idx]], color="#dc2626", s=70, zorder=6)
        ax_ts.annotate(
            f"Max Load: {stats['max_g']:.2f} g @ {df['time_s'].iloc[pk_idx]:.1f} s",
            xy=(df["time_s"].iloc[pk_idx], df["accel_vert_g"].iloc[pk_idx]),
            xytext=(10, -16), textcoords="offset points",
            fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff7ed", edgecolor="#ea580c")
        )
        ax_ts.set_ylabel("Vertical Acceleration [g]", fontsize=9.5)
        ax_ts.set_title(f"Flight Dataset: Acceleration vs. Time ({fs_hz:.0f} Hz)", fontsize=11, fontweight="bold", pad=10)

    ax_ts.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_ts.legend(loc="upper right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    # Summary Banner + Dataset Preview Table
    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Dataset Mode:</b> <span style="color:#7dd3fc;font-weight:700;">{dataset_mode}</span> &nbsp;|&nbsp;
            <b>Samples:</b> {stats['samples']} &nbsp;|&nbsp;
            <b>Max Alt:</b> {stats['max_alt_m']:.1f} m &nbsp;|&nbsp;
            <b>Peak / Min g:</b> {stats['max_g']:.2f} g / {stats['min_g']:.2f} g &nbsp;|&nbsp;
            <b>Saved CSV:</b> <code>{csv_filename}</code>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;margin-top:4px;line-height:1.5;">
            {FLIGHT_PROFILES[profile_name]['desc']}
        </div>
    </div>
    """))

    display(df.head(8).style.hide(axis="index"))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_prof = widgets.Dropdown(options=list(FLIGHT_PROFILES.keys()), value="Hand Launch & Thermal Glide", description="Flight Profile:", style=lbl_style, layout=ctrl_layout)
w_mode = widgets.Dropdown(options=["Altitude Dataset [m]", "Acceleration Dataset [g]"], value="Altitude Dataset [m]", description="Dataset Output:", style=lbl_style, layout=ctrl_layout)
w_fs   = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=10.0, description="Sample Rate [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_dur  = widgets.FloatSlider(value=25.0, min=5.0,  max=90.0,  step=5.0,  description="Duration [s]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_alt0 = widgets.FloatSlider(value=35.0, min=5.0,  max=150.0, step=5.0,  description="Launch Alt [m]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_nois = widgets.FloatSlider(value=1.0,  min=0.0,  max=3.0,   step=0.2,  description="Sensor Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_log = widgets.Button(description="Log Flight Dataset", button_style="primary", icon="database", layout=widgets.Layout(width="210px", height="36px"))
out_log = widgets.Output(layout=widgets.Layout(width="100%"))
run_counter = {"count": 0}

def refresh_log(*args):
    run_counter["count"] += 1
    with out_log:
        clear_output(wait=True)
        draw_glider_rig_and_dataset(
            w_prof.value, w_mode.value, w_fs.value, w_dur.value,
            w_alt0.value, w_nois.value, run_counter["count"]
        )

btn_log.on_click(refresh_log)
for c in [w_prof, w_mode, w_fs, w_dur, w_alt0, w_nois]:
    c.observe(refresh_log, names="value")

header_log = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Model Glider Low-Cost Data Logger &amp; Flight Dataset</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Convert model glider flights into time-series altitude or acceleration datasets with 2D trajectory view
        </div>
    </div>
""")

controls_log = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_prof, w_mode, w_fs], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_dur, w_alt0, w_nois, btn_log], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_log, controls_log, out_log)
refresh_log()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. GLIDER FLIGHT PROFILES & LOGGER ENGINE
# ==========================================
FLIGHT_PROFILES = {
    "Hand Launch & Thermal Glide": {
        "climb_gain_m": 28.0,
        "cruise_mps": 9.5,
        "thermal_boost": 5.5,
        "desc": "Starts from 0 m, climbs on launch/lift, rides a mild thermal pocket, and glides smoothly back to 0 m touchdown."
    },
    "Steep Dive & Pull-Up": {
        "climb_gain_m": 35.0,
        "cruise_mps": 12.0,
        "thermal_boost": 0.0,
        "desc": "Climbs from 0 m, enters a controlled dive with higher speed, pulls up (g-load peak), and lands back at 0 m."
    },
    "Gusty Ridge Glide": {
        "climb_gain_m": 30.0,
        "cruise_mps": 10.5,
        "thermal_boost": 3.0,
        "desc": "Climbs from 0 m and glides under periodic wind gusts and vertical acceleration variations down to 0 m."
    },
}

def generate_glider_dataset(profile_name, fs_hz, duration_s, peak_alt_m, noise_level, seed=42):
    prof = FLIGHT_PROFILES[profile_name]
    n_samples = max(int(fs_hz * duration_s), 60)
    t = np.linspace(0.0, duration_s, n_samples)
    dt = t[1] - t[0]
    u = t / max(duration_s, 1e-6)  # normalized time [0 -> 1]

    # Forward speed & horizontal distance
    v_fwd = prof["cruise_mps"] + 1.2 * np.sin(np.pi * u)
    if profile_name == "Steep Dive & Pull-Up":
        dive_window = np.exp(-((u - 0.52) / 0.11)**2)
        v_fwd = v_fwd + 6.5 * dive_window
    x_dist = np.cumsum(v_fwd) * dt

    # Smooth trajectory starting at 0 m, rising to peak_alt_m, then gliding back to 0 m
    # Envelope: 0 at u=0, peaks around u=0.35..0.45, returns to 0 at u=1
    base_envelope = (u ** 0.65) * ((1.0 - u) ** 1.15)
    base_envelope = base_envelope / max(np.max(base_envelope), 1e-9)
    alt_clean = peak_alt_m * base_envelope

    if profile_name == "Hand Launch & Thermal Glide":
        thermal = prof["thermal_boost"] * np.exp(-((u - 0.52) / 0.14)**2) * np.sin(np.pi * u)
    elif profile_name == "Steep Dive & Pull-Up":
        dive_dip = -0.18 * peak_alt_m * np.exp(-((u - 0.56) / 0.09)**2)
        pull_lift = 0.06 * peak_alt_m * np.exp(-((u - 0.68) / 0.09)**2)
        thermal = (dive_dip + pull_lift) * np.sin(np.pi * u)
    else:  # "Gusty Ridge Glide"
        thermal = prof["thermal_boost"] * np.sin(2.0 * np.pi * u) * np.sin(np.pi * u)

    alt_clean = np.maximum(alt_clean + thermal, 0.0)
    # Enforce exact 0.0 m at start and touchdown
    alt_clean[0] = 0.0
    alt_clean[-1] = 0.0

    # Vertical velocity & vertical acceleration [g]
    vz = np.gradient(alt_clean, dt)
    az_mps2 = np.gradient(vz, dt)
    a_vert_g = 1.0 + (az_mps2 / 9.81)

    if profile_name == "Steep Dive & Pull-Up":
        a_vert_g += 1.45 * np.exp(-((u - 0.64) / 0.07)**2)
    elif profile_name == "Gusty Ridge Glide":
        a_vert_g += 0.35 * np.sin(3.5 * t) + 0.22 * np.cos(7.1 * t)

    # Sensor noise (kept near zero right at launch/touchdown ground contact)
    rng = np.random.default_rng(seed)
    ground_mask = np.sin(np.pi * u) ** 0.35
    alt_meas = np.maximum(alt_clean + rng.normal(0.0, 0.30 * noise_level, size=n_samples) * ground_mask, 0.0)
    alt_meas[0] = 0.0
    alt_meas[-1] = 0.0

    accel_g_meas = a_vert_g + rng.normal(0.0, 0.06 * noise_level, size=n_samples)

    df = pd.DataFrame({
        "time_s": np.round(t, 3),
        "distance_m": np.round(x_dist, 2),
        "altitude_m": np.round(alt_meas, 2),
        "accel_vert_g": np.round(accel_g_meas, 3),
    })

    return df, {
        "max_alt_m": float(np.max(alt_meas)),
        "final_alt_m": float(alt_meas[-1]),
        "total_dist_m": float(x_dist[-1]),
        "max_g": float(np.max(accel_g_meas)),
        "min_g": float(np.min(accel_g_meas)),
        "samples": n_samples,
    }

# ==========================================
# 2. 2D FLIGHT PATH + SELECTED DATASET PLOT
# ==========================================
def draw_glider_rig_and_dataset(profile_name, dataset_mode, fs_hz, duration_s, peak_alt_m, noise_level, run_idx):
    df, stats = generate_glider_dataset(
        profile_name, fs_hz, duration_s, peak_alt_m, noise_level, seed=42 + run_idx
    )
    csv_filename = f"glider_flight_{profile_name.lower().replace(' ', '_').replace('&', 'and')}.csv"
    df.to_csv(csv_filename, index=False)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_path, ax_ts) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.05, 1.4]})

    # --- LEFT: 2D Glider Flight Profile (0 m -> Climb/Glide -> 0 m) ---
    ax_path.plot(df["distance_m"], df["altitude_m"], color="#2563eb", lw=2.3, label="Glider Trajectory")
    ax_path.fill_between(df["distance_m"], 0.0, df["altitude_m"], color="#93c5fd", alpha=0.25)
    ax_path.scatter([df["distance_m"].iloc[0]], [0.0], color="#16a34a", s=70, zorder=6, label="Launch (0 m)")
    ax_path.scatter([df["distance_m"].iloc[-1]], [0.0], color="#dc2626", s=70, zorder=6, label="Touchdown (0 m)")

    ax_path.set_title(f"2D Flight Profile — {profile_name}", fontsize=11, fontweight="bold", pad=10)
    ax_path.set_xlabel("Downrange Distance [m]", fontsize=9.5)
    ax_path.set_ylabel("Altitude [m]", fontsize=9.5)
    ax_path.set_ylim(bottom=0.0)
    ax_path.legend(loc="upper right", fontsize=8, frameon=True, facecolor="white")

    # --- RIGHT: Selected Flight Dataset Time Series ---
    if dataset_mode == "Altitude Dataset [m]":
        ax_ts.plot(df["time_s"], df["altitude_m"], color="#0284c7", lw=2.0, label="Logged Altitude [m]")
        pk_idx = int(np.argmax(df["altitude_m"]))
        ax_ts.scatter([df["time_s"].iloc[pk_idx]], [df["altitude_m"].iloc[pk_idx]], color="#dc2626", s=70, zorder=6)
        ax_ts.annotate(
            f"Peak Alt: {stats['max_alt_m']:.1f} m @ {df['time_s'].iloc[pk_idx]:.1f} s",
            xy=(df["time_s"].iloc[pk_idx], df["altitude_m"].iloc[pk_idx]),
            xytext=(12, -16), textcoords="offset points",
            fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#f0f9ff", edgecolor="#0284c7")
        )
        ax_ts.set_ylabel("Altitude [m]", fontsize=9.5)
        ax_ts.set_ylim(bottom=0.0)
        ax_ts.set_title(f"Flight Dataset: Altitude vs. Time ({fs_hz:.0f} Hz)", fontsize=11, fontweight="bold", pad=10)
    else:
        ax_ts.plot(df["time_s"], df["accel_vert_g"], color="#ea580c", lw=1.8, label="Vertical Acceleration [g]")
        ax_ts.axhline(1.0, color="#64748b", ls="--", lw=1.1, label="1.0 g Level")
        pk_idx = int(np.argmax(df["accel_vert_g"]))
        ax_ts.scatter([df["time_s"].iloc[pk_idx]], [df["accel_vert_g"].iloc[pk_idx]], color="#dc2626", s=70, zorder=6)
        ax_ts.annotate(
            f"Max Load: {stats['max_g']:.2f} g @ {df['time_s'].iloc[pk_idx]:.1f} s",
            xy=(df["time_s"].iloc[pk_idx], df["accel_vert_g"].iloc[pk_idx]),
            xytext=(-135, -14), textcoords="offset points",
            fontsize=8.5, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#fff7ed", edgecolor="#ea580c")
        )
        ax_ts.set_ylabel("Vertical Acceleration [g]", fontsize=9.5)
        ax_ts.set_title(f"Flight Dataset: Acceleration vs. Time ({fs_hz:.0f} Hz)", fontsize=11, fontweight="bold", pad=10)

    ax_ts.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_ts.legend(loc="upper right", fontsize=8.5, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:13px 17px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="font-size:13px;">
            <b>Dataset Mode:</b> <span style="color:#7dd3fc;font-weight:700;">{dataset_mode}</span> &nbsp;|&nbsp;
            <b>Samples:</b> {stats['samples']} &nbsp;|&nbsp;
            <b>Max Alt:</b> {stats['max_alt_m']:.1f} m &nbsp;|&nbsp;
            <b>Peak / Min g:</b> {stats['max_g']:.2f} g / {stats['min_g']:.2f} g &nbsp;|&nbsp;
            <b>Saved CSV:</b> <code>{csv_filename}</code>
        </div>
        <div style="font-size:12.5px;color:#cbd5e1;margin-top:4px;line-height:1.5;">
            {FLIGHT_PROFILES[profile_name]['desc']}
        </div>
    </div>
    """))

    display(df.head(8).style.hide(axis="index"))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "145px"}
ctrl_layout = widgets.Layout(width="340px")

w_prof = widgets.Dropdown(options=list(FLIGHT_PROFILES.keys()), value="Hand Launch & Thermal Glide", description="Flight Profile:", style=lbl_style, layout=ctrl_layout)
w_mode = widgets.Dropdown(options=["Altitude Dataset [m]", "Acceleration Dataset [g]"], value="Altitude Dataset [m]", description="Dataset Output:", style=lbl_style, layout=ctrl_layout)
w_fs   = widgets.FloatSlider(value=50.0, min=10.0, max=200.0, step=10.0, description="Sample Rate [Hz]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_dur  = widgets.FloatSlider(value=30.0, min=5.0,  max=90.0,  step=5.0,  description="Duration [s]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_peak = widgets.FloatSlider(value=45.0, min=10.0, max=150.0, step=5.0,  description="Peak Alt [m]:", readout_format=".0f", style=lbl_style, layout=ctrl_layout)
w_nois = widgets.FloatSlider(value=1.0,  min=0.0,  max=3.0,   step=0.2,  description="Sensor Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_log = widgets.Button(description="Log Flight Dataset", button_style="primary", icon="database", layout=widgets.Layout(width="210px", height="36px"))
out_log = widgets.Output(layout=widgets.Layout(width="100%"))
run_counter = {"count": 0}

def refresh_log(*args):
    run_counter["count"] += 1
    with out_log:
        clear_output(wait=True)
        draw_glider_rig_and_dataset(
            w_prof.value, w_mode.value, w_fs.value, w_dur.value,
            w_peak.value, w_nois.value, run_counter["count"]
        )

btn_log.on_click(refresh_log)
for c in [w_prof, w_mode, w_fs, w_dur, w_peak, w_nois]:
    c.observe(refresh_log, names="value")

header_log = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Model Glider Low-Cost Data Logger &amp; Flight Dataset</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Convert model glider flights into time-series altitude or acceleration datasets with 2D trajectory view
        </div>
    </div>
""")

controls_log = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_prof, w_mode, w_fs], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_dur, w_peak, w_nois, btn_log], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_log, controls_log, out_log)
refresh_log()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))